In [ ]:
import torch
import numpy as np
from torch.nn import *
from inspect import stack
from math import prod


class F(Module):
    def __init__(s, f, *_, name="", **__):
        super().__init__()
        s.f = f
        s.name = name or (f.__name__ if getattr(f, "__name__", "<lambda>") != "<lambda>" else stack()[1].function)
        s._ = _
        s.__ = __
        for _ in enumerate(_), __.items():
            for __, _ in _:
                if isinstance(_, Module):
                    s.add_module(str(__), _)
                elif isinstance(_, Parameter):
                    s.register_parameter(str(__), _)

    def forward(s, x):
        try:
            return s.f(x, *s._, **s.__)
        except:
            s.name = "\33[31m" + s.name + "\33[0m"
            raise

    def _get_name(s):
        return s.name


def Seq(*_, f=lambda x, *_: [x := _(x) for _ in _][-1]):
    return F(f, *[_ for _ in _ if not isinstance(_, Identity) for _ in (_._ if isinstance(_, F) and _.f == f else [_])]) if len(_) > 1 else _[0] if _ else Identity()


def Map(*_, f=lambda x, *_: [_(x) for _ in _]):
    return F(f, *_)


def Add(*_):
    return Seq(Map(*_), F(sum))


def Mul(*_):
    return Seq(Map(*_), F(prod))


def Lift(*_):
    return Seq(F(torch.transpose, -2, -1), *_, F(torch.transpose, -2, -1))


def LiftSz(s, *_):
    return F(lambda x, _: _(x.view(-1, *s)).view(*x.shape[:-1], -1), Seq(*_))


@F
def linspace(x):
    return torch.linspace(0, 1, x.size(-2))[:, None]


def Pos(h, *_):
    return Add(Seq(*_), Seq(linspace, Linear(1, h, False)))


@F
def trispace(x):
    return torch.stack([torch.sin(x := torch.linspace(0, 2 * torch.pi, x.size(-2))), torch.cos(x)], 1)


def TriPos(h, *_):
    return Add(Seq(*_), Seq(trispace, Linear(2, h, False)))


def Stride(x, r=0, s=1):
    if not r:
        return F(Stride, x, s)
    x = functional.pad(x, (0, 0, r - 1, r - 1))
    return x.as_strided(x.shape[:-2] + ((x.size(-2) - r) // s + 1, r, x.size(-1)), x.stride()[:-2] + (s * x.stride(-2),) + x.stride()[-2:])


def Unstride(x):
    return x.as_strided(x.shape[:-3] + (x.size(-3) - x.size(-2) + 1,) + x.shape[-2:], x.stride()[:-2] + (x.stride(-2) - 1, x.stride(-1)), (x.size(-2) - 1) * x.stride(-2)).mean(-2)


def Unfold1d(x, r=0, **_):
    if not r:
        return F(Unfold1d, x, **_)
    return functional.unfold(x.transpose(1, 2)[..., None], (r, 1), **_).unflatten(1, (x.size(2), -1)).permute(0, 3, 2, 1)


def Fold1d(x, l, r=0, **_):
    if not r:
        return F(Fold1d, x, l, **_)
    return functional.fold(x.permute(0, 3, 2, 1).flatten(1, 2), [l, 1], [r, 1], **_)[..., 0].transpose(1, 2)


def Nn(i, h, o=0, bias=True, pos=lambda h, _: _, bn=Identity, act=ReLU(True)):
    return Seq(pos(h, (Linear if i else Identity)(i, h, bn == Identity)), bn(h, track_running_stats=False), act, (Linear if o else Identity)(h, o))


def SwiGLU(i, h, o=0, bias=True, pos=lambda h, _: _, bn=Identity, act=Hardswish(True)):
    return Seq(Mul(Seq(pos(h, (Linear if i else Identity)(i, h, bn == Identity)), bn(h, track_running_stats=False), act, name="swi"), Linear(i, h)), (Linear if o else Identity)(h, o))


@F
def AttentionMin(x):
    x = (x if isinstance(x, list) else [x]) * 3
    return (x[0] @ x[1].transpose(-2, -1)).softmax(-1) @ x[2]


def Attention(i, h, o=0, nn=[Nn] * 4, bias=False):
    return Seq(Map(nn[0](i[0], h, h, bias), nn[1](i[1], h, bias and h), nn[2](i[2], h)), AttentionMin, *[nn[3](0, h, o)] if o else [])


@F
def MultiheadAttentionMin(x):
    x = (x if isinstance(x, list) else [x]) * 3
    return torch.einsum("...Llh,...lhk->...Lhk", torch.einsum("...Lhk,...lhk->...Llh", x[0], x[1]).softmax(-2), x[2])


def MultiheadAttention(i, h, o=0, head=1, nn=[Nn] * 4, bias=False):
    return Seq(
        Map(
            Seq(nn[0](i[0], head * h, head * h, bias), Unflatten(-1, [head, h]), name="q"),
            Seq(nn[1](i[1], head * h, head * h, bias), Unflatten(-1, [head, h]), name="k"),
            Seq(nn[2](i[2], head * h, head * h), Unflatten(-1, [head, h]), name="v"),
        ),
        MultiheadAttentionMin,
        Flatten(-2),
        *[nn[3](0, head * h, o)] if o else [],
    )

In [ ]:
from dataclasses import dataclass
from IPython.display import clear_output
import matplotlib.pyplot as plt


def Try(f, *_, **__):
    try:
        return f(*_, **__)
    except:
        pass


def plot(plot={}, fig=(12, 8), **dat):
    plt.figure(figsize=fig)
    plt.tight_layout()
    root = None
    line = {}
    for i, (name, dat) in enumerate(dat.items()):
        prefix = name.split("_")[0]
        lin = line.setdefault(prefix, [])
        if lin:
            ax = lin[0].axes
        else:
            if root:
                ax = root.twinx()
                ax.spines.right.set_position(("axes", 1 + (len(line) - 2) * 0.12))
            else:
                ax = root = plt.subplot()
            ax.set(ylabel=prefix)
        lin.append(ax.plot(*dat, **{"c": f"C{i}", "label": name, **plot.get(prefix, {}), **plot.get(name, {})})[0])
    root.legend(handles=sum(line.values(), []))
    return line


class Log(dict):
    i = 1

    def __init__(s, it=10, **_):
        s.it = it
        s._ = _

    def __call__(s, **dat):
        for name, dat in dat.items():
            if dat is not None:
                _ = s.get(name)
                if not _:
                    _ = s[name] = [np.empty(9999, "f"), np.empty([9999, Try(len, dat)] if Try(len, dat) else 9999, "f"), 0, []]
                _[0][_[2] + len(_[3])] = s.i
                _[3].append(dat)
        if not s.i % s.it:
            s.plt()
        s.i += 1

    def sync(s):
        for s in s.values():
            s[1][s[2] : s[2] + len(s[3])] = s[3]
            s[2] += len(s[3])
            s[3].clear()

    def plt(s):
        s.sync()
        plot(**s._, **{i: [s[0][: s[2]], s[1][: s[2]]] for i, s in s.items()})
        clear_output(True)
        plt.show()


@dataclass
class AutoLR:
    o: torch.optim.Optimizer
    exp: float = 0.999
    mom: float = 0.9
    up: float = 1 / 0.999
    down: float = 0.999
    til: float = 1e-5
    los = 0

    def step(s, los):
        s.o.param_groups[0]["lr"] *= s.up if los < s.los * s.exp else s.down
        s.los = (s.los or los) * s.mom + los * (1 - s.mom)
        return s.o.param_groups[0]["lr"] < s.til

In [ ]:
m = 64
m = Seq(
    TriPos(m, Linear(1, m, False)),
    Unflatten(1, (-1, 16)),
    *(
        Seq(
            Add(
                Seq(
                    Nn(m, m, m, pos=TriPos, act=SiLU(True)),
                    AttentionMin,
                ),
                Linear(m, m, False),
            ),
            Add(
                Seq(
                    F(torch.transpose, 1, 2),
                    Nn(m, m, m, pos=TriPos, act=SiLU(True)),
                    AttentionMin,
                    F(torch.transpose, 1, 2),
                ),
                Linear(m, m, False),
            ),
        )
        for i in range(3)
    ),
    Flatten(1, 2),
    Nn(m, m, 256, pos=TriPos, act=SiLU(True)),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.001, 500000, pct_start=0.1, div_factor=9, final_div_factor=999)

In [ ]:
log = Log()
while O.last_epoch < O.total_steps:
    x = torch.rand(64, 256)
    y = m(x[..., None])
    x = x.argsort()
    l = functional.cross_entropy(y.transpose(1, 2), x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    if not (O.last_epoch + 1) % 100:
        log(los=l.detach(), acc=None if (O.last_epoch + 1) % 1000 else (y.argmax(2) == x).sum() / x.numel(), lr=o.param_groups[0]["lr"])